In [3]:
import pandas as pd
import numpy as np
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer


import numpy as np

In [5]:
df = pd.read_csv('dataset_recsys_telegramm_bot2.csv')

In [6]:
df

,Unnamed: 0,id,genres_list,artist_name,name_song,main_genre,final_text
0,0,5YuEm3H0E6oeGxuUPYmqpA,"['adult standards', 'contemporary vocal jazz',...",Diana Krall,Departure Bay,Pop,fade scent summertime arbutus tree firs gliste...
1,1,1dNRYPfHxhp8mx3JgzhG2L,"['contemporary r&b', 'hip hop', 'hip pop', 'ne...",Dru Hill,Away,Hip-Hop,away away away listen start together love jour...
2,2,1NKMm7HrVBxrV6js3iZwzL,['viral rap'],Insane Clown Posse,Truth Dare,Hip-Hop,truth dare double dare promise repeat truth da...
3,3,1zlcdKTUUxbUFIlrZUgyox,"['alternative hip hop', 'east coast hip hop', ...",Masta Ace,I Did It,Hip-Hop,yeah yeah right yeah yeah yeah love respect ca...
4,4,5x9PXqjZIwk7NxrLV6NVC2,"['grunge pop', 'pop rock']",Splender,Save It For Later,Rock,thinkin tomorrow simple things think might lik...
...,...,...,...,...,...,...,...
16090,16090,55lsFTXePg7yLpmjeE27Lp,"['alternative metal', 'nu metal', 'rap metal',...",Papa Roach,Not That Beautiful,Hip-Hop,pray fast run past pick phone find alone wear ...
16091,16091,7stgamMva4CcykLigaf5UD,"['dance pop', 'pop', 'r&b']",Beyoncé,Resentment,Pop,wish could believe alright everything tell rea...
16092,16092,3sfAumzy0rGj6WEhuuRqz6,"['g funk', 'gangster rap', 'hip hop', 'hyphy',...",Spice 1,Strap On The Side,Hip-Hop,rollin muthafuckin strap side fuck east bay ro...
16093,16093,7vknUoCF2ZqUnxAagtq1ho,['alternative r&b'],Reverie,Just Wanna Love You,Rock,baby come home ive wait night wanna love kiss ...


In [7]:
# 1. Используем твою уже готовую матрицу X (TF-IDF)
tfidf = TfidfVectorizer(max_features=5000, stop_words='english', min_df=3)

X = tfidf.fit_transform(df['final_text'])

svd = TruncatedSVD(n_components=2, random_state=42)
X_svd = svd.fit_transform(X)


# 2. Строим SVD на 50 компонент (это стандарт для LSA в таких задачах)
svd_rec = TruncatedSVD(n_components=50, random_state=42)
X_svd_rec = svd_rec.fit_transform(X)

print(f"Матрица для графиков: {X_svd.shape} (у тебя уже есть)")
print(f"Матрица для рекомендаций: {X_svd_rec.shape} (мы сделали сейчас)")

Матрица для графиков: (16095, 2) (у тебя уже есть)
Матрица для рекомендаций: (16095, 50) (мы сделали сейчас)


In [8]:
!pip install spotipy --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 354.2/354.2 kB 7.5 MB/s eta 0:00:00


In [9]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from IPython.display import Image, display, HTML

In [10]:
# ВСТАВЬ СВОИ КЛЮЧИ
cid = '0c3036484fa94a4aa50c76530cdecfc5'
secret = 'dfad42355e7f48f4a632333c5cd40e76'


client_credentials_manager = SpotifyClientCredentials(client_id=cid, client_secret=secret)
sp = spotipy.Spotify(client_credentials_manager=client_credentials_manager)

def get_spotify_data(artist, song_name):
    try:
        # Ищем трек
        q = f"artist:{artist} track:{song_name}"
        results = sp.search(q=q, type='track', limit=1)

        if results['tracks']['items']:
            track = results['tracks']['items'][0]
            cover_url = track['album']['images'][0]['url'] # Картинка 640x640
            preview_url = track['preview_url'] # Ссылка на mp3 (может быть None)
            spotify_link = track['external_urls']['spotify']
            return cover_url, spotify_link
        else:
            return None, None
    except Exception as e:
        return None, None

In [11]:
def recommend_music(song_title, method='svd', top_n=5, enrich=True):
    # 1. Ищем индекс песни в базе
    # (для надежности приводим к нижнему регистру при поиске)
    try:
        idx = df[df['name_song'].str.lower() == song_title.lower()].index[0]
    except IndexError:
        return f"Песня '{song_title}' не найдена в датасете."

    print(f"Ищем похожие на: {df.iloc[idx]['artist_name']} - {df.iloc[idx]['name_song']} [{method.upper()}]")

    # 2. Выбираем матрицу в зависимости от метода
    if method == 'svd':
        matrix = X_svd_rec
    elif method == 'cosine':
        matrix = X # Твоя TF-IDF матрица
    elif method == 'hybrid':
        # Смешиваем: 70% смысла (SVD) + 30% точности слов (TF-IDF)
        # Нужно считать косинусное расстояние отдельно, но для скорости
        # давай просто возьмем SVD, он "умнее"
        matrix = X_svd_rec

    # 3. Считаем косинусную близость
    # Берем вектор нужной песни
    query_vec = matrix[idx].reshape(1, -1)
    # Считаем similarity со всеми
    sim_scores = cosine_similarity(query_vec, matrix).flatten()

    # 4. Сортируем
    # argsort возвращает индексы от меньшего к большему, берем хвост, разворачиваем
    # top_n + 1, чтобы исключить саму песню (она будет первой)
    top_indices = sim_scores.argsort()[-(top_n+1):][::-1]

    # Исключаем саму песню из выдачи
    top_indices = [i for i in top_indices if i != idx]

    # 5. Собираем результаты
    results = []
    for i in top_indices:
        row = df.iloc[i]
        score = sim_scores[i]

        cover, link = None, None
        if enrich:
            cover, link = get_spotify_data(row['artist_name'], row['name_song'])

        results.append({
            'Artist': row['artist_name'],
            'Song': row['name_song'],
            'Genre': row['main_genre'],
            'Similarity': round(score, 3),
            'Cover': cover,
            'Link': link
        })

    return pd.DataFrame(results)

# Функция для красивого отображения картинок в ноутбуке
def show_results(res_df):
    def path_to_image_html(path):
        if path:
            return '<img src="'+ path + '" width="60" >'
        return 'No Image'

    def create_link(url):
        if url:
            return f'<a href="{url}" target="_blank">Listen</a>'
        return ''

    display(HTML(res_df.to_html(escape=False, formatters=dict(Cover=path_to_image_html, Link=create_link))))

In [12]:
# --- ПРИМЕР СРАВНЕНИЯ ---
target_song = "Burning Inside" # Из твоего скриншота (Ministry)

print("--- METHOD: TF-IDF (Cosine) ---")
print("Ищет точные совпадения слов (literal matching)")
res_cosine = recommend_music(target_song, method='cosine', top_n=5, enrich=True)
show_results(res_cosine)

print("\n--- METHOD: SVD (LSA) ---")
print("Ищет совпадение смыслов и контекста (semantic matching)")
res_svd = recommend_music(target_song, method='svd', top_n=5, enrich=True)
show_results(res_svd)

# --- АНАЛИТИКА: ПЕРЕСЕЧЕНИЕ ---
set_cosine = set(res_cosine['Song'])
set_svd = set(res_svd['Song'])

intersection = set_cosine.intersection(set_svd)
print(f"\nОбщие песни в обоих методах: {len(intersection)}")
if len(intersection) > 0:
    print(intersection)
else:
    print("Методы нашли абсолютно разные песни! Это показывает, что SVD нашел скрытые связи.")

--- METHOD: TF-IDF (Cosine) ---
Ищет точные совпадения слов (literal matching)
Ищем похожие на: Ministry - Burning Inside [COSINE]


,Artist,Song,Genre,Similarity,Cover,Link
0,STRFKR,Burnin' Up,Rock,0.689,,Listen
1,Hundred Suns,The Prestaliis II,Rock,0.649,,Listen
2,Tina Arena,Burn,Pop,0.573,,Listen
3,Mako,Beam - Dannic Mix,Pop,0.526,,Listen
4,Slipknot,Me Inside,Hip-Hop,0.524,,Listen



--- METHOD: SVD (LSA) ---
Ищет совпадение смыслов и контекста (semantic matching)
Ищем похожие на: Ministry - Burning Inside [SVD]


,Artist,Song,Genre,Similarity,Cover,Link
0,Toxik,Social Overload,Rock,0.899,,Listen
1,Heaven's Basement,I Am Electric,Rock,0.851,,Listen
2,AFI,Lower It,Rock,0.828,,Listen
3,Hundred Suns,The Prestaliis II,Rock,0.817,,Listen
4,Lucinda Williams,Burning Bridges,Rock,0.816,,Listen



Общие песни в обоих методах: 1
{'The Prestaliis II'}


In [1]:
!pip install pyTelegramBotAPI --quiet
import telebot
from telebot import types

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.3/48.3 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.8/294.8 kB 7.9 MB/s eta 0:00:00


In [14]:
# --- НАСТРОЙКИ ---
# Вставь сюда свой рабочий токен
BOT_TOKEN = '8520302688:AAH-Me-Bwah2l3mqzcymQtAIRqj7vcrlknY'
bot = telebot.TeleBot(BOT_TOKEN)

# Словарь для хранения состояния: {user_id: chosen_song_index}
user_state = {}

# --- НОВАЯ ФУНКЦИЯ ПОИСКА ---
def search_in_db(query):
    query = query.lower().strip()

    # Ищем вхождение запроса ЛИБО в исполнителе, ЛИБО в названии песни
    # case=False делает поиск нечувствительным к регистру
    mask = df['name_song'].str.lower().str.contains(query, regex=False) | \
           df['artist_name'].str.lower().str.contains(query, regex=False)

    results = df[mask]
    return results

# --- ВСПОМОГАТЕЛЬНАЯ ФУНКЦИЯ: ПОКАЗАТЬ ВЫБОР АЛГОРИТМА ---
def show_algo_selection(chat_id, song_idx):
    # Сохраняем выбор пользователя
    user_state[chat_id] = song_idx

    row = df.loc[song_idx]
    artist = row['artist_name']
    song = row['name_song']

    # Пробуем достать обложку
    cover_url, _ = get_spotify_data(artist, song)

    markup = types.InlineKeyboardMarkup()
    btn_svd = types.InlineKeyboardButton("🧠 По смыслу (SVD)", callback_data='method_svd')
    btn_cos = types.InlineKeyboardButton("📝 По словам (Cosine)", callback_data='method_cosine')
    markup.add(btn_svd, btn_cos)

    caption = f"Выбрано: **{artist} - {song}**\n\nКак будем искать похожие?"

    if cover_url:
        bot.send_photo(chat_id, cover_url, caption=caption, parse_mode='Markdown', reply_markup=markup)
    else:
        bot.send_message(chat_id, caption, parse_mode='Markdown', reply_markup=markup)


# --- ОБРАБОТЧИКИ СООБЩЕНИЙ ---

@bot.message_handler(commands=['start'])
def send_welcome(message):
    bot.reply_to(message, "Привет! 🎧\n"
                          "Напиши мне **Исполнителя** или **Название песни**.\n"
                          "Пример: `Eminem` или `Italian Leather Sofa`")

@bot.message_handler(content_types=['text'])
def handle_text(message):
    query = message.text
    results = search_in_db(query)

    # Сценарий 1: Ничего не нашли
    if len(results) == 0:
        bot.reply_to(message, "😔 Ничего не найдено. Проверь название.")
        return

    # Сценарий 2: Нашли ровно одну песню
    if len(results) == 1:
        idx = results.index[0]
        show_algo_selection(message.chat.id, idx)
        return

    # Сценарий 3: Нашли много песен (например, по запросу исполнителя)
    # Показываем кнопки (максимум 10, чтобы не засорять чат)
    top_results = results.head(10)

    markup = types.InlineKeyboardMarkup()
    for idx, row in top_results.iterrows():
        # Формируем текст кнопки: "Artist - Song"
        btn_text = f"{row['artist_name']} - {row['name_song']}"
        # В callback_data передаем 'pick_<ID_строки>'
        markup.add(types.InlineKeyboardButton(btn_text, callback_data=f"pick_{idx}"))

    bot.send_message(message.chat.id, f"Нашел {len(results)} треков. Выбери нужный:", reply_markup=markup)


@bot.callback_query_handler(func=lambda call: True)
def callback_query(call):
    chat_id = call.message.chat.id

    # 1. ОБРАБОТКА ВЫБОРА ПЕСНИ ИЗ СПИСКА
    if call.data.startswith('pick_'):
        song_idx = int(call.data.split('_')[1])
        bot.answer_callback_query(call.id) # Убираем часики
        # Удаляем предыдущее сообщение со списком кнопок, чтобы было чисто
        try:
            bot.delete_message(chat_id, call.message.message_id)
        except:
            pass
        # Переходим к выбору алгоритма
        show_algo_selection(chat_id, song_idx)
        return

    # 2. ОБРАБОТКА ВЫБОРА АЛГОРИТМА (SVD/Cosine)
    if chat_id not in user_state:
        bot.answer_callback_query(call.id, "Сессия истекла, введи поиск заново.")
        return

    song_idx = user_state[chat_id]
    method = call.data

    bot.answer_callback_query(call.id, "Считаю математику...")

    # Выбор матрицы
    if method == 'method_svd':
        matrix = X_svd_rec
        algo_name = "SVD (Смыслы)"
    else:
        matrix = X
        algo_name = "Cosine (Слова)"

    # Расчет
    query_vec = matrix[song_idx].reshape(1, -1)
    sim_scores = cosine_similarity(query_vec, matrix).flatten()

    # Топ-5 похожих
    top_indices = sim_scores.argsort()[-(6):][::-1]
    top_indices = [i for i in top_indices if i != song_idx][:5]

    # Формирование ответа
    response_text = f"💿 **Рекомендации ({algo_name}):**\n\n"

    for i in top_indices:
        row = df.iloc[i]
        score = round(sim_scores[i], 3)
        _, sp_link = get_spotify_data(row['artist_name'], row['name_song'])
        link_str = f"[Spotify]({sp_link})" if sp_link else "—"

        response_text += f"🎵 *{row['artist_name']} - {row['name_song']}*\n"
        response_text += f"   Жанр: {row['main_genre']} (Сходство: {score})\n"
        response_text += f"   🔗 {link_str}\n\n"

    bot.send_message(chat_id, response_text, parse_mode='Markdown', disable_web_page_preview=True)

# --- ЗАПУСК ---
print("Бот обновлен и запущен...")
try:
    bot.polling(none_stop=True)
except Exception as e:
    print(f"Ошибка: {e}")

Бот обновлен и запущен...
Ошибка: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))


МОЖНО ВЗЯТЬ ОЦЕНКИ ПОЛЬЗОВАТЕЛЕЙ И МОДЕЛИ ДООБУЧАЮТСЯ